# 01 — Auditoria do AZT1D e protocolo experimental v0

Este notebook implementa a primeira etapa prática do projeto: validar se os CSVs do AZT1D podem alimentar um experimento de previsão glicêmica de **60 minutos de histórico → 30 minutos à frente**.

Ele ainda **não treina uma LSTM**. Primeiro verifica o esquema, qualidade temporal, disponibilidade de variáveis e número de janelas candidatas. Isso evita medir um modelo com dados desalinhados ou vazamento temporal.

## Como usar

1. Abra este notebook a partir da pasta `notebooks/` ou da raiz do projeto.
2. Execute as células na ordem apresentada.
3. Leia a tabela de testes antes de passar ao pré-processamento. Lacunas e duplicatas são esperadas no AZT1D; elas devem ser tratadas por uma política explícita, não ignoradas.

Dependências obrigatórias: `pandas` e `numpy`. `matplotlib` é opcional: sem ele as tabelas e os testes continuam funcionando, mas o gráfico de exemplo é pulado.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print('matplotlib não está instalado; as tabelas serão geradas, mas o gráfico será pulado.')

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

In [ ]:
# Parâmetros do protocolo v0. Eles são deliberadamente explícitos para poderem ser
# revisados antes do primeiro treinamento.
GRID_FREQUENCY = '5min'
HISTORY_MINUTES = 60
HORIZON_MINUTES = 30
MAX_GAP_MINUTES = 15
SAVE_REPORTS = True

GRID_MINUTES = int(pd.Timedelta(GRID_FREQUENCY).total_seconds() / 60)
assert HISTORY_MINUTES % GRID_MINUTES == 0
assert HORIZON_MINUTES % GRID_MINUTES == 0
HISTORY_STEPS = HISTORY_MINUTES // GRID_MINUTES
HORIZON_STEPS = HORIZON_MINUTES // GRID_MINUTES
WINDOW_SPAN_STEPS = HISTORY_STEPS + HORIZON_STEPS

def find_project_root(start: Path = Path.cwd()) -> Path:
    # Encontra a raiz sem depender do diretório em que o notebook foi aberto.
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'AZT1D 2025' / 'CGM Records').is_dir():
            return candidate
    raise FileNotFoundError(
        'Não encontrei AZT1D 2025/CGM Records. Abra o notebook dentro do repositório.'
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'AZT1D 2025' / 'CGM Records'
REPORT_DIR = PROJECT_ROOT / 'results' / 'data_audit'

def subject_number(path: Path) -> int:
    match = re.search(r'\d+', path.name)
    if match is None:
        raise ValueError(f'Não foi possível extrair o identificador de {path.name!r}.')
    return int(match.group())

subject_paths = {}
for subject_dir in sorted(DATA_DIR.glob('Subject *'), key=subject_number):
    csv_files = list(subject_dir.glob('*.csv'))
    if len(csv_files) != 1:
        raise ValueError(f'{subject_dir} deveria conter exatamente um CSV; encontrados: {csv_files}')
    subject_paths[subject_number(subject_dir)] = csv_files[0]

expected_subjects = set(range(1, 26))
found_subjects = set(subject_paths)
print(f'Raiz do projeto: {PROJECT_ROOT}')
print(f'Participantes encontrados: {len(found_subjects)}')
print('Participantes ausentes:', sorted(expected_subjects - found_subjects) or 'nenhum')
print(f'Grade proposta: {GRID_FREQUENCY}; histórico: {HISTORY_STEPS} passos; horizonte: {HORIZON_STEPS} passos.')

In [ ]:
# O Subject 14 usa outro nome para a leitura de glicose e outra ordem de colunas.
# A padronização é feita pelo NOME da coluna, nunca pela posição no arquivo.
EXPECTED_COLUMNS = [
    'EventDateTime', 'DeviceMode', 'BolusType', 'Basal',
    'CorrectionDelivered', 'TotalBolusInsulinDelivered',
    'FoodDelivered', 'CarbSize', 'CGM',
]
NUMERIC_COLUMNS = [
    'Basal', 'CorrectionDelivered', 'TotalBolusInsulinDelivered',
    'FoodDelivered', 'CarbSize', 'CGM',
]
COLUMN_ALIASES = {
    'Readings (CGM / BGM)': 'CGM',
}

def load_subject(subject: int, csv_path: Path):
    raw = pd.read_csv(csv_path)
    raw.columns = raw.columns.str.strip()
    original_columns = raw.columns.tolist()
    raw = raw.rename(columns=COLUMN_ALIASES)

    duplicated_column_names = raw.columns[raw.columns.duplicated()].tolist()
    missing_columns = [column for column in EXPECTED_COLUMNS if column not in raw.columns]

    # reindex mantém o notebook executável e deixa as colunas ausentes explícitas no relatório.
    frame = raw.reindex(columns=EXPECTED_COLUMNS).copy()
    frame['EventDateTime'] = pd.to_datetime(frame['EventDateTime'], errors='coerce')
    numeric_parse_errors = {}
    for column in NUMERIC_COLUMNS:
        raw_values = frame[column]
        converted = pd.to_numeric(raw_values, errors='coerce')
        numeric_parse_errors[column] = int(
            (raw_values.notna() & raw_values.astype(str).str.strip().ne('') & converted.isna()).sum()
        )
        frame[column] = converted
    # Não corrigir escala de basal nesta fase: conservar o valor bruto para auditoria.
    frame['Basal_raw'] = frame['Basal']

    schema_info = {
        'subject': subject,
        'source_file': str(csv_path.relative_to(PROJECT_ROOT)),
        'original_columns': ' | '.join(original_columns),
        'missing_columns': ' | '.join(missing_columns),
        'duplicated_column_names': ' | '.join(duplicated_column_names),
        'used_alias_for_cgm': 'Readings (CGM / BGM)' in original_columns,
        'numeric_parse_errors': ' | '.join(f'{column}={count}' for column, count in numeric_parse_errors.items() if count),
    }
    return frame, schema_info

In [ ]:
def audit_subject(subject: int, frame: pd.DataFrame) -> dict:
    ordered = frame.sort_values('EventDateTime', kind='stable').reset_index(drop=True)
    timestamps = ordered['EventDateTime']
    positive_deltas = timestamps.diff().dt.total_seconds().div(60)
    positive_deltas = positive_deltas[positive_deltas > 0]

    cgm = ordered['CGM']
    valid_cgm = cgm.dropna()
    repeated_timestamp_cgm = ordered.dropna(subset=['EventDateTime']).groupby('EventDateTime')['CGM'].nunique(dropna=True)
    cadence_share = (
        float(np.isclose(positive_deltas.to_numpy(), GRID_MINUTES).mean())
        if not positive_deltas.empty else np.nan
    )

    return {
        'subject': subject,
        'rows': len(ordered),
        'start': timestamps.min(),
        'end': timestamps.max(),
        'coverage_days': (timestamps.max() - timestamps.min()).total_seconds() / 86_400
            if timestamps.notna().any() else np.nan,
        'invalid_timestamps': int(timestamps.isna().sum()),
        'duplicate_timestamps': int(timestamps.duplicated().sum()),
        'duplicate_timestamps_with_cgm_conflict': int((repeated_timestamp_cgm > 1).sum()),
        'median_interval_min': positive_deltas.median(),
        'max_interval_min': positive_deltas.max(),
        'share_exactly_5min': cadence_share,
        'gaps_over_15min': int((positive_deltas > MAX_GAP_MINUTES).sum()),
        'cgm_present': int(cgm.notna().sum()),
        'cgm_missing': int(cgm.isna().sum()),
        'cgm_min': valid_cgm.min(),
        'cgm_max': valid_cgm.max(),
        'cgm_at_40': int(cgm.eq(40).sum()),
        'cgm_at_400': int(cgm.eq(400).sum()),
        'cgm_nonpositive': int(cgm.le(0).sum()),
        'cgm_above_400': int(cgm.gt(400).sum()),
        'basal_present': int(ordered['Basal'].notna().sum()),
        'basal_over_10': int(ordered['Basal_raw'].gt(10).sum()),
        'bolus_present': int(ordered['TotalBolusInsulinDelivered'].notna().sum()),
        'bolus_positive': int(ordered['TotalBolusInsulinDelivered'].gt(0).sum()),
        'carb_present': int(ordered['CarbSize'].notna().sum()),
        'carb_positive': int(ordered['CarbSize'].gt(0).sum()),
        'carb_only_zero_or_missing': bool(ordered['CarbSize'].fillna(0).eq(0).all()),
    }

subject_frames = {}
schema_rows = []
audit_rows = []

for subject, csv_path in subject_paths.items():
    frame, schema_info = load_subject(subject, csv_path)
    subject_frames[subject] = frame
    schema_rows.append(schema_info)
    audit_rows.append(audit_subject(subject, frame))

schema_df = pd.DataFrame(schema_rows).sort_values('subject').reset_index(drop=True)
audit_df = pd.DataFrame(audit_rows).sort_values('subject').reset_index(drop=True)

print('Resumo temporal e de disponibilidade por participante:')
audit_df

In [ ]:
print('Validação do esquema por participante:')
schema_df[['subject', 'missing_columns', 'duplicated_column_names', 'used_alias_for_cgm', 'source_file']]

In [ ]:
# Testes que precisam passar antes de gerar janelas de treino.
# Duplicatas e gaps não impedem a auditoria, mas exigem uma regra explícita no pré-processamento.
subjects_missing_columns = schema_df.loc[schema_df['missing_columns'].ne(''), 'subject'].tolist()
invalid_timestamp_count = int(audit_df['invalid_timestamps'].sum())
subjects_without_cgm = audit_df.loc[audit_df['cgm_present'].eq(0), 'subject'].tolist()
duplicate_timestamp_count = int(audit_df['duplicate_timestamps'].sum())
gap_count = int(audit_df['gaps_over_15min'].sum())

checks = pd.DataFrame([
    {
        'check': '25 participantes esperados encontrados',
        'passed': found_subjects == expected_subjects,
        'detail': f'encontrados={len(found_subjects)}, ausentes={sorted(expected_subjects - found_subjects)}',
    },
    {
        'check': 'esquema padronizado possui todas as colunas esperadas',
        'passed': schema_df['missing_columns'].eq('').all(),
        'detail': f'participantes com colunas ausentes: {subjects_missing_columns}',
    },
    {
        'check': 'timestamps válidos em todas as linhas',
        'passed': audit_df['invalid_timestamps'].eq(0).all(),
        'detail': f'linhas inválidas: {invalid_timestamp_count}',
    },
    {
        'check': 'ao menos uma leitura CGM em cada participante',
        'passed': audit_df['cgm_present'].gt(0).all(),
        'detail': f'participantes sem CGM: {subjects_without_cgm}',
    },
    {
        'check': 'duplicatas temporais foram identificadas para tratamento',
        'passed': True,
        'detail': f'total de timestamps duplicados: {duplicate_timestamp_count}',
    },
    {
        'check': f'lacunas acima de {MAX_GAP_MINUTES} min foram identificadas para tratamento',
        'passed': True,
        'detail': f'total de lacunas: {gap_count}',
    },
])
checks['status'] = np.where(checks['passed'], 'PASSOU', 'REVISAR')
checks[['status', 'check', 'detail']]

## Teste de janelas candidatas de CGM

A célula abaixo usa apenas CGM para responder: *há blocos contínuos suficientes para 12 observações de entrada e um alvo seis passos depois?*

Em vez de escolher arbitrariamente a última leitura quando dois valores de CGM caem no mesmo bin de 5 minutos, ela marca o bin como conflito e o exclui das janelas candidatas. Ela também não define ainda a regra final de agregação de insulina e carboidratos.

In [ ]:
def cgm_grid_and_window_count(frame: pd.DataFrame) -> tuple[pd.DataFrame, int, dict]:
    ordered = frame.dropna(subset=['EventDateTime']).sort_values('EventDateTime', kind='stable')
    values = ordered[['EventDateTime', 'CGM']].copy()
    values['grid_time'] = values['EventDateTime'].dt.floor(GRID_FREQUENCY)
    grouped = values.groupby('grid_time', sort=True)['CGM']

    # Bins com duas ou mais leituras CGM diferentes são ambíguos. Não calcular média,
    # mediana ou última leitura: a próxima etapa deve decidir uma regra clínica se preciso.
    binned = pd.concat([
        grouped.size().rename('records_in_bin'),
        grouped.nunique(dropna=True).rename('cgm_distinct_values'),
        grouped.first().rename('CGM'),
    ], axis=1)
    binned.loc[binned['cgm_distinct_values'].gt(1), 'CGM'] = np.nan
    full_index = pd.date_range(binned.index.min(), binned.index.max(), freq=GRID_FREQUENCY)
    cgm_grid = binned.reindex(full_index)
    cgm_grid.index.name = 'EventDateTime'
    cgm_grid['records_in_bin'] = cgm_grid['records_in_bin'].fillna(0).astype(int)
    cgm_grid['cgm_distinct_values'] = cgm_grid['cgm_distinct_values'].fillna(0).astype(int)
    cgm_grid['cgm_conflict'] = cgm_grid['cgm_distinct_values'].gt(1)
    cgm_grid['valid_for_cgm_window'] = cgm_grid['CGM'].notna() & ~cgm_grid['cgm_conflict']

    contiguous_observations = cgm_grid['valid_for_cgm_window'].astype(int).rolling(
        WINDOW_SPAN_STEPS, min_periods=WINDOW_SPAN_STEPS
    ).sum().eq(WINDOW_SPAN_STEPS)
    run_id = cgm_grid['valid_for_cgm_window'].ne(cgm_grid['valid_for_cgm_window'].shift()).cumsum()
    clean_segment_lengths = cgm_grid['valid_for_cgm_window'].groupby(run_id).sum()
    clean_segment_lengths = clean_segment_lengths[clean_segment_lengths.gt(0)]
    grid_stats = {
        'bins_with_multiple_records': int(cgm_grid['records_in_bin'].gt(1).sum()),
        'bins_with_cgm_conflict': int(cgm_grid['cgm_conflict'].sum()),
        'clean_segments': int(len(clean_segment_lengths)),
        'longest_clean_segment_steps': int(clean_segment_lengths.max()) if not clean_segment_lengths.empty else 0,
    }
    return cgm_grid, int(contiguous_observations.sum()), grid_stats

window_rows = []
cgm_grids = {}
for subject, frame in subject_frames.items():
    cgm_grid, candidate_windows, grid_stats = cgm_grid_and_window_count(frame)
    cgm_grids[subject] = cgm_grid
    window_rows.append({
        'subject': subject,
        'grid_rows': len(cgm_grid),
        'grid_cgm_missing_or_ambiguous': int((~cgm_grid['valid_for_cgm_window']).sum()),
        **grid_stats,
        'candidate_cgm_windows': candidate_windows,
    })

window_df = pd.DataFrame(window_rows).sort_values('subject').reset_index(drop=True)
print(
    f'Uma janela candidata exige {HISTORY_STEPS} passos de histórico e {HORIZON_STEPS} passos até o alvo '
    f'({WINDOW_SPAN_STEPS} observações CGM contínuas no total).'
)
window_df

In [ ]:
# Gráfico de inspeção visual. Troque o participante ou o início do intervalo quando necessário.
PLOT_SUBJECT = min(subject_frames)
plot_frame = subject_frames[PLOT_SUBJECT].dropna(subset=['EventDateTime']).sort_values('EventDateTime')
plot_start = plot_frame['EventDateTime'].min().floor('D')
plot_end = plot_start + pd.Timedelta(days=1)
day = plot_frame.loc[plot_frame['EventDateTime'].between(plot_start, plot_end)].copy()

if HAS_MATPLOTLIB:
    fig, ax_glucose = plt.subplots(figsize=(14, 4))
    ax_glucose.plot(day['EventDateTime'], day['CGM'], color='tab:blue', linewidth=1.5, label='CGM')
    ax_glucose.set_title(f'Participante {PLOT_SUBJECT}: primeiras 24 horas disponíveis')
    ax_glucose.set_xlabel('Horário local do Arizona')
    ax_glucose.set_ylabel('Glicose')
    ax_glucose.grid(alpha=0.25)

    carbs = day.dropna(subset=['CarbSize'])
    bolus = day.dropna(subset=['TotalBolusInsulinDelivered'])
    ax_events = ax_glucose.twinx()
    if not carbs.empty:
        ax_events.scatter(carbs['EventDateTime'], carbs['CarbSize'], marker='^', color='tab:orange', label='Carboidrato')
    if not bolus.empty:
        ax_events.scatter(bolus['EventDateTime'], bolus['TotalBolusInsulinDelivered'], marker='v', color='tab:red', label='Bolus')
    ax_events.set_ylabel('Carboidrato / bolus registrados')

    handles_1, labels_1 = ax_glucose.get_legend_handles_labels()
    handles_2, labels_2 = ax_events.get_legend_handles_labels()
    ax_glucose.legend(handles_1 + handles_2, labels_1 + labels_2, loc='upper right')
    fig.autofmt_xdate()
    plt.show()
else:
    print('Para gerar este gráfico, instale matplotlib no ambiente do notebook.')

In [ ]:
protocol_v0 = pd.DataFrame([
    ('Frequência-alvo', GRID_FREQUENCY),
    ('Histórico de entrada', f'{HISTORY_MINUTES} min ({HISTORY_STEPS} passos)'),
    ('Horizonte de previsão', f'{HORIZON_MINUTES} min ({HORIZON_STEPS} passos)'),
    ('Alvo', 'CGM no horizonte futuro'),
    ('Entradas previstas', 'CGM, insulina e carboidratos — após regra de agregação documentada'),
    ('Split populacional', 'por participante; paciente de teste nunca entra no treino populacional'),
    ('Split de personalização', 'cronológico: início para fine-tuning, período posterior intocado para teste'),
    ('Métricas iniciais', 'MAE e RMSE'),
    ('Política provisória para gaps', f'não criar janela que atravesse ausência em grade de {GRID_FREQUENCY}'),
], columns=['item', 'decisão v0'])
protocol_v0

In [ ]:
duplicate_count = int(audit_df['duplicate_timestamps'].sum())
conflicting_bin_count = int(window_df['bins_with_cgm_conflict'].sum())
gap_count = int(audit_df['gaps_over_15min'].sum())
censored_cgm_count = int(audit_df['cgm_at_40'].sum() + audit_df['cgm_at_400'].sum())
large_basal_count = int(audit_df['basal_over_10'].sum())
carb_unavailable_subjects = audit_df.loc[audit_df['carb_only_zero_or_missing'], 'subject'].tolist()
zero_window_subjects = window_df.loc[window_df['candidate_cgm_windows'].eq(0), 'subject'].tolist()

quality_flags = pd.DataFrame([
    ('Bins CGM com conflito', 'REVISAR' if conflicting_bin_count else 'PASSOU', conflicting_bin_count, 'Excluir esses bins e suas janelas; não agregar leituras divergentes.'),
    ('Timestamps duplicados', 'REVISAR' if duplicate_count else 'PASSOU', duplicate_count, 'Definir como registros idênticos serão consolidados no pré-processamento.'),
    (f'Lacunas acima de {MAX_GAP_MINUTES} min', 'REVISAR' if gap_count else 'PASSOU', gap_count, 'Manter a quebra de janela; não interpolar CGM na v0.'),
    ('Valores CGM nos limites 40/400', 'REVISAR' if censored_cgm_count else 'PASSOU', censored_cgm_count, 'Sinalizar possível censura/saturação, sem excluir silenciosamente.'),
    ('Valores de basal acima de 10', 'REVISAR' if large_basal_count else 'PASSOU', large_basal_count, 'Preservar Basal_raw; não corrigir escala sem documentação.'),
    ('Participantes sem carboidrato positivo', 'REVISAR' if carb_unavailable_subjects else 'PASSOU', carb_unavailable_subjects, 'Não interpretar zero como ausência de refeição sem validar o campo.'),
    ('Participantes sem janela CGM limpa', 'FALHOU' if zero_window_subjects else 'PASSOU', zero_window_subjects, 'Excluir ou revisar participantes sem janelas elegíveis.'),
], columns=['item', 'status', 'valor', 'ação necessária'])
quality_flags

In [ ]:
if SAVE_REPORTS:
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    audit_df.to_csv(REPORT_DIR / 'subject_audit.csv', index=False)
    schema_df.to_csv(REPORT_DIR / 'schema_audit.csv', index=False)
    window_df.to_csv(REPORT_DIR / 'candidate_windows_cgm.csv', index=False)
    checks.to_csv(REPORT_DIR / 'validation_checks.csv', index=False)
    protocol_v0.to_csv(REPORT_DIR / 'protocol_v0.csv', index=False)
    quality_flags.to_csv(REPORT_DIR / 'quality_flags.csv', index=False)
    print(f'Relatórios salvos em: {REPORT_DIR.relative_to(PROJECT_ROOT)}')

hard_failures = checks.loc[~checks['passed'], 'check'].tolist()
if hard_failures:
    print('Antes do pré-processamento, revise:', hard_failures)
else:
    print('Auditoria estrutural concluída. Próximo passo: implementar a regra de agregação e o gerador de janelas.')

review_items = quality_flags.loc[quality_flags['status'].eq('REVISAR'), 'item'].tolist()
if review_items:
    print('Decisões que precisam ser documentadas antes do treino:', review_items)

print('Não use ainda as janelas candidatas para treinar: primeiro defina como basal, bolus e carboidratos serão agregados na grade de 5 min.')

## Critério para avançar

Avance para o notebook de pré-processamento quando os quatro testes estruturais estiverem em `PASSOU` e a equipe tiver documentado: (a) a regra para registros duplicados, (b) a política para gaps, (c) a agregação de basal/bolus/carboidratos e (d) o conjunto de participantes de treino, validação e teste.

A próxima implementação deve produzir janelas sem vazamento e, antes da LSTM, comparar a previsão de persistência (`CGM atual`) contra o alvo a 30 minutos.